# CT-RATE Lumora Training

This notebook uses the same two-phase DenseNet121 + GPT-2 pipeline as `lumora_train_local.ipynb`, adapted for CT-RATE NIfTI chest CT volumes and radiology report CSVs. CT-RATE is gated and very large, so accept the terms on Hugging Face first and start with a small subset.

In [1]:
from pathlib import Path

DATA_DIR = Path("data/ct_rate")
CHECKPOINT_DIR = Path("checkpoints/ct_rate")

BATCH_SIZE = 2
MAX_TEXT_LENGTH = 128
NUM_WORKERS = 0
PHASE1_EPOCHS = 3
PHASE2_EPOCHS = 3
PHASE1_LR = 1e-4
PHASE2_LR = 2e-5

print("Configuration loaded")
print(f"DATA_DIR: {DATA_DIR.resolve()}")
print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR.resolve()}")

Configuration loaded
DATA_DIR: /Users/md.nurealamsiddiquee/Projects/lumora/data/ct_rate
CHECKPOINT_DIR: /Users/md.nurealamsiddiquee/Projects/lumora/checkpoints/ct_rate


## Download

Run this after accepting the gated dataset terms. The defaults below download metadata plus a tiny subset of volumes for a smoke test. Remove or raise the limits only when you are ready for the storage cost.

In [ ]:
!uv run python ct_rate_train_local.py download --data-dir {DATA_DIR} --limit-train 20 --limit-valid 5

## Data Pipeline Smoke Test

This checks that CT-RATE report rows match local NIfTI files and produce the same image/text tensors used by the Lumora model.

In [ ]:
from transformers import AutoTokenizer
from ct_rate_train_local import get_device, build_entries, CTRateReportDataset

device = get_device()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

train_entries, train_csv = build_entries(DATA_DIR, "train")
train_dataset = CTRateReportDataset(train_entries[:8], tokenizer, MAX_TEXT_LENGTH)
sample = train_dataset[0]

print(f"Device: {device}")
print(f"Train CSV: {train_csv}")
print(f"Train samples with local files: {len(train_dataset):,}")
print(f"Image tensor shape: {tuple(sample['image'].shape)}")
print(f"Input IDs shape: {tuple(sample['input_ids'].shape)}")

## Train

Remove `--limit-train` and `--limit-val` when you are ready to train on all downloaded files.

In [ ]:
!uv run python ct_rate_train_local.py train \
  --data-dir {DATA_DIR} \
  --checkpoint-dir {CHECKPOINT_DIR} \
  --batch-size {BATCH_SIZE} \
  --max-text-length {MAX_TEXT_LENGTH} \
  --num-workers {NUM_WORKERS} \
  --phase1-epochs {PHASE1_EPOCHS} \
  --phase1-lr {PHASE1_LR} \
  --phase2-epochs {PHASE2_EPOCHS} \
  --phase2-lr {PHASE2_LR} \
  --limit-train 16 \
  --limit-val 4